# Synthetic valley advanced model

This notebook builds the coastal synthetic-valley model with the MODFLOW 6 advanced packages, adds two multi-aquifer wells, runs it, and compares the simulated heads to the true heads at the observation wells. It then asks how pumping from the well near the stream changes the water the stream carries out of the model. The companion notebook [`synthetic-valley-a-base`](synthetic-valley-a-base.ipynb) runs the same valley with simple boundary packages.

> **Model reference.** The synthetic valley model is described in Hughes, J.D., Langevin, C.D., Paulinski, S.R., Larsen, J.D., and Brakenhoff, D., 2024, FloPy workflows for creating structured and unstructured MODFLOW models: Groundwater, v. 62, no. 1, p. 124–139, https://doi.org/10.1111/gwat.13327.

## What this notebook covers

By the end of this notebook you will be able to:

- say what the Unsaturated Zone Flow, Streamflow Routing, Lake, Water Mover, and Multi-Aquifer Well packages represent,
- build the Multi-Aquifer Well package with `flopy.mf6.ModflowGwfmaw()`,
- plot the water table in plan view and in cross section and compare simulated heads to observed heads,
- read the water budgets of the model and the lake and plot them as pie charts and yearly bars, and
- measure how much a pumping well reduces the discharge of a nearby stream.

## The model

The synthetic valley drains south to the sea. The model uses a structured grid of 40 rows and 25 columns of 152.4 meter (m) cells, and five layers: layers 1 and 2 are the upper aquifer, layer 3 is a clay confining unit present only in part of the valley, and layers 4 and 5 are the lower aquifer. Where there is no clay, the layer 3 cells are **vertical pass-through cells** (`idomain` of -1), which have no volume and connect layer 2 directly to layer 4. The top of layer 1 is the land surface, and layer 1 is convertible, so its saturated thickness follows the water table. The sea is a General-Head Boundary (**GHB**) at a head of 0 m along the southern edge.

The **advanced packages** represent the surface-water features and the land surface as more than a fixed boundary:

- **Unsaturated Zone Flow (UZF)** moves rainfall down through the unsaturated zone to the water table and takes evapotranspiration from it. Where the water table reaches the land surface, the infiltration it cannot accept runs off.
- **Streamflow Routing (SFR)** routes water down the stream, reach by reach, and exchanges water with the aquifer beneath each reach. A **reach** is a length of stream in one cell.
- **Lake (LAK)** keeps a water balance for the lake, so its level rises and falls with rainfall, evaporation, runoff, and leakage to and from the aquifer. The lake sits on top of the aquifer and leaks through its bed to the layer 1 cell beneath it. Where the lake bed is at the bottom of layer 1, the layer 1 cell is a pass-through cell and the lake leaks to layer 2. A table gives the volume and surface area of the lake at each lake level, from the deepest point of its bed up to the level at which it would spill over the ridge to the south, so the lake has no outlet.
- **Drains (DRN)** on the layer 1 cells away from the lake and the stream remove groundwater wherever the water table rises to within 0.15 m (0.5 feet) of the land surface, as seepage does.
- **Water Mover (MVR)** sends part of the runoff from UZF, and all of the water the drains remove, to where the land surface drains: the lake north of the ridge, or the stream reach the water reaches south of it. Water that drains straight to the coast leaves the model.
- **Multi-Aquifer Well (MAW)** represents a well screened across several layers. It solves for one water level in the wellbore and works out how much water each layer gives up.

The first stress period is a 16,071-day **steady-state** period, which gives the flow system before pumping. Twenty one-year transient periods follow. MODFLOW has no built-in units, so every input shares one length and one time unit: here meters and days.

## Imports

Import FloPy, Matplotlib, NumPy, and pandas. `sv_notebook_helpers` holds the functions that build the model, read the true and simulated heads at the observation wells, and read and save the stream discharge.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sv_notebook_helpers import (
    build_advanced_without_maw,
    record_sfr_discharge,
    sfr_discharge,
    simulated_heads,
    true_heads,
)

## Build the model with the advanced packages

`build_advanced_without_maw()` writes the base model to a new workspace with the Recharge, Evapotranspiration, River, and Well packages replaced by UZF, SFR, LAK, and MVR, and loads it with `flopy.mf6.MFSimulation.load()`. It also takes the land surface, the hydraulic conductivity, the starting heads, and the drains from the advanced model, because under the lake layer 1 is aquifer rather than the very high hydraulic conductivity cells of the base model. It leaves out MAW, which you build next. `pathlib.Path` builds the path to the workspace.

In [ ]:
name = "sv"
new_ws = pl.Path("models/synthetic-valley-advanced")

sim = build_advanced_without_maw(new_ws)
gwf = sim.get_model(name)
gwf.get_package_list()

Read the arrays the wells and figures need: `idomain` and the cell bottoms `botm` from the discretization (**DIS**) package. Each is a NumPy array of shape `(nlay, nrow, ncol)`, and FloPy counts layers, rows, and columns from 0, so `botm[3]` is the bottom of layer 4.

The lake cells are the layer 1 cells under the lake, the cells listed in the connection data of the Lake package. The cross sections below run through the row and column at the center of the lake.

In [ ]:
idomain = gwf.dis.idomain.array
botm = gwf.dis.botm.array
extent = gwf.modelgrid.extent
nper = sim.tdis.nper.data

# the lake cells are the layer 1 cells the lake is connected to
lake = np.zeros(idomain.shape[1:], dtype=bool)
for cellid in gwf.lak.connectiondata.array["cellid"]:
    lake[cellid[1], cellid[2]] = True
lake_rows, lake_cols = np.nonzero(lake)
row_xs = int(round(lake_rows.mean()))
col_xs = int(round(lake_cols.mean()))
print(
    f"{lake.sum()} lake cells; sections along row {row_xs + 1} and column {col_xs + 1}"
)

## Build the Multi-Aquifer Well package

Two wells pump from the lower aquifer, each screened across layers 4 and 5. The `reilly` well is in the north of the valley, and the `vc` well is three columns (457 m) west of the stream. Set the pumping rate of each well in cubic meters per day (m³/d). A negative rate removes water from the aquifer.

In [ ]:
reilly_rate = -1897.2  # m3/d
vc_rate = -7588.9  # m3/d

# well name: (first layer, last layer, row, column), counted from 0
well_data = {
    "reilly": (3, 4, 5, 14),
    "vc": (3, 4, 32, 5),
}
rates = {"reilly": reilly_rate, "vc": vc_rate}

The package data gives one row per well: its number (counted from 0), the wellbore radius, the bottom of the well, the starting water level, the equation used for the conductance between the well and the aquifer, the number of layers it is connected to, and its name. `"thiem"` computes that conductance from the cell's hydraulic conductivity and the Thiem equation for steady radial flow to a well.

In [ ]:
# (well, radius, bottom, starting head, conductance equation, connections, name)
# (0, 0.1524, -67.2, 0.0, "thiem", 2, "reilly")
package_data = []
for iw, (well, (k0, k1, i, j)) in enumerate(well_data.items()):
    package_data.append((iw, 0.1524, float(botm[k1, i, j]), 0.0, "thiem", 2, well))
package_data

The connection data gives one row per well per layer: the well and connection numbers, the cell `(layer, row, column)`, and the top and bottom of the screen in that layer. Each screen runs the full thickness of its cell, from the bottom of the layer above to the bottom of the layer. The skin values are not used by the Thiem equation, so they are set to -999.

In [ ]:
# (well, connection, (layer, row, column), screen top, screen bottom, skin K, skin radius)
connection_data = []
for iw, (well, (k0, k1, i, j)) in enumerate(well_data.items()):
    for ic, k in enumerate(range(k0, k1 + 1)):
        screen_top = float(botm[k - 1, i, j])
        screen_bottom = float(botm[k, i, j])
        connection_data.append(
            (iw, ic, (k, i, j), screen_top, screen_bottom, -999.0, -999.0)
        )
connection_data

The period data switches the wells off for the steady-state period and on, at their pumping rates, from the second stress period. It is a dictionary keyed by the stress-period number, counted from 0. A setting carries forward to later stress periods until it is changed, so the rates only need to be given once.

In [ ]:
maw_spd = {
    0: [(iw, "status", "inactive") for iw in range(len(well_data))],
    1: [(iw, "status", "active") for iw in range(len(well_data))]
    + [(iw, "rate", rates[well]) for iw, well in enumerate(well_data)],
}

maw = flopy.mf6.ModflowGwfmaw(
    gwf,
    pname="pwell",
    boundnames=True,
    nmawwells=len(well_data),
    packagedata=package_data,
    connectiondata=connection_data,
    perioddata=maw_spd,
)
gwf.get_package_list()

## Run the model

Ask LAK to save its own water budget to a CSV file by setting `budgetcsv_filerecord`; the water budget section below reads it. Then write the input files with `write_simulation()` and run MODFLOW 6 with `run_simulation()`, which returns `True` when the run ends normally.

In [ ]:
gwf.lak.budgetcsv_filerecord = "sv.lak.budget.csv"
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

## Water table at the end of the steady-state period

Read the heads at the end of the steady-state period with `gwf.output.head().get_data()`. `kstpkper=(0, 0)` selects the first time step of the first stress period, counting from 0. Inactive cells hold a head of 1.0E+30, which `masked_values` leaves out of the plots.

LAK computes the lake level, its **stage**: read it with `gwf.lak.output.stage().get_data()`. The map shows the stage in the lake cells, so the lake surface appears as one flat level. Under the lake, layer 1 is aquifer beneath the lake bed, so the sections show its head and draw the lake water above it, between the lake bed and the stage.

Map the head in layer 1, which is the water table, and draw it along a west-east and a north-south cross section through the center of the lake. In the cross sections, `plot_array(head, head=head)` colors each cell only up to the simulated head, so the colored area ends at the water table. The sections show the top 25 m of the model, from the highest land surface down to the bottom of the upper aquifer.

In [ ]:
width = 17.15 / 2.54  # figure width, inches
aquifer_head = gwf.output.head().get_data(kstpkper=(0, 0))
stage = gwf.lak.output.stage().get_data(kstpkper=(0, 0)).ravel()[0]
# the map shows the lake surface
head = aquifer_head.copy()
head[0][lake] = stage
water_table = np.ma.masked_greater(head[0], 1.0e29)
vmin, vmax = np.floor(water_table.min()), np.ceil(water_table.max())
levels = np.arange(vmin, vmax + 0.5, 0.5)
top = gwf.dis.top.array
x_ns = gwf.modelgrid.xcellcenters[0, col_xs]
y_ew = gwf.modelgrid.ycellcenters[row_xs, 0]
# elevation range of the sections: land surface to 25 m below its highest point
ylim = (np.ceil(top.max()) - 25.0, np.ceil(top.max()))
sections = {
    "B": ({"row": row_xs}, "West-east section", "Distance from the west edge (m)"),
    "C": (
        {"column": col_xs},
        "North-south section",
        "Distance from the north edge (m)",
    ),
}

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"], ["A", "C"]],
        figsize=(width, 0.85 * width),
        layout="constrained",
        width_ratios=[1.0, 1.5],
    )
    ax = axd["A"]
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    pa = mm.plot_array(head, masked_values=[1.0e30], vmin=vmin, vmax=vmax)
    cs = mm.contour_array(
        head, masked_values=[1.0e30], levels=levels, colors="black", linewidths=0.5
    )
    ax.clabel(cs, fmt="%g", fontsize=6)
    mm.contour_array(lake.astype(float), levels=[0.5], colors="blue", linewidths=1.0)
    mm.plot_bc(package=gwf.sfr, color="cyan")
    mm.plot_bc(package=gwf.ghb, color="purple", kper=0)
    mm.plot_bc(package=maw, color="red", kper=1)
    ax.plot(extent[:2], [y_ew, y_ew], color="red", lw=1.0, ls="--")
    ax.plot([x_ns, x_ns], extent[2:], color="red", lw=1.0, ls="--")
    ax.text(extent[0], y_ew, "B ", color="red", ha="right", va="center")
    ax.text(x_ns, extent[3], "C", color="red", ha="center", va="bottom")
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    flopy.plot.styles.heading(ax=ax, letter="A", heading="Water table")
    fig.colorbar(pa, ax=ax, orientation="horizontal", shrink=0.9, label="Head (m)")

    for letter, (line, heading, xlabel) in sections.items():
        ax = axd[letter]
        xs = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line=line)
        xs.plot_array(
            aquifer_head,
            head=aquifer_head,
            masked_values=[1.0e30],
            vmin=vmin,
            vmax=vmax,
        )
        # the lake, between its bed and its stage
        if "row" in line:
            edges = np.r_[0.0, np.cumsum(gwf.dis.delr.array)]
            wet, bed = lake[row_xs], top[row_xs]
        else:
            edges = np.r_[0.0, np.cumsum(gwf.dis.delc.array)]
            wet, bed = lake[:, col_xs], top[:, col_xs]
        for n in np.nonzero(wet)[0]:
            ax.fill_between(edges[n : n + 2], bed[n], stage, color="lightblue", lw=0)
        xs.plot_grid(lw=0.25, color="0.5")
        xs.plot_surface(top, color="saddlebrown", lw=1.0)
        xs.plot_surface(water_table, color="blue", lw=1.0)
        ax.set_ylim(ylim)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Elevation (m)")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)
    handles = [
        mpl.lines.Line2D([], [], color="saddlebrown", lw=1.0, label="Land surface"),
        mpl.patches.Patch(color="lightblue", label="Lake"),
        mpl.lines.Line2D([], [], color="blue", lw=1.0, label="Water table"),
    ]
    flopy.plot.styles.graph_legend(
        ax=axd["C"],
        handles=handles,
        labels=[h.get_label() for h in handles],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.3),
        ncol=3,
    )

In [ ]:
aquifer = np.ma.masked_where(lake, water_table)
print(f"water table from {aquifer.min():.2f} to {aquifer.max():.2f} m")
print(f"lake stage {stage:.2f} m")

**What to look for.** The water table (A) falls from 6.98 m in the northeast to -0.13 m in the stream cells nearest the coast, where the streambed is below sea level. The lake stage is 5.97 m. The lake loses 1.26 million cubic meters per year to the aquifer beneath the rest of its bed and gains 0.45 million cubic meters per year from the aquifer beneath its northern and northeastern part, where the water table is above the stage. The contours bend upstream along the stream (cyan), which gains water from the upper aquifer along its length. In the sections (B and C) the lake bed falls from the shore to the bottom of layer 1, the water table (blue) is close to the land surface (brown) toward the coast, and it is several meters below the land surface on the high ground east of the lake and on the ridge south of it. The wells (red) are switched off in this period, so the heads are those of the valley before pumping.

## Compare the simulated heads to the true heads

The observation wells measure the water table (layer 1) and the lower aquifer (layers 4 and 5) at 17 locations. The true heads come from the Voronoi-grid advanced model in `examples/data/synthetic-valley/data`, which defines the truth for the class. `true_heads()` returns them and `simulated_heads()` reads the simulated heads the model saved at the same wells, each as a table with one column per well and one row per output time. Select the end of the steady-state period with `.loc[]`.

A **residual** is the simulated head minus the true head. A positive residual means the model is too high.

In [ ]:
t_steady = gwf.output.head().get_times()[0]
kinds = {"A": ("wt", "Water table"), "B": ("aq", "Lower aquifer")}
residuals = {}
for letter, (kind, label) in kinds.items():
    true = true_heads(kind).loc[t_steady]
    sim_head = simulated_heads(new_ws, kind).loc[t_steady, true.index]
    r = sim_head - true
    residuals[letter] = (true, sim_head, r)
    rmse = np.sqrt((r**2).mean())
    print(
        f"{label}: true heads {true.min():.3f} to {true.max():.3f} m; "
        f"mean residual {r.mean():.3f} m, root-mean-square {rmse:.3f} m, "
        f"largest {r.abs().max():.3f} m"
    )

Map the residuals at each well, colored from red (simulated head too low) to blue (too high) and sized by the absolute residual, so the largest misfits stand out, and plot the simulated heads with the true heads on a one-to-one plot, where a well whose simulated head equals its true head falls on the diagonal line. The observation names hold the row and column of each well counted from 0, so `aq05-i14-j11` is in row 15 and column 12; `obs_cell()` reads them back.

In [ ]:
def obs_cell(obsname):
    # "AQ05-I14-J11" -> (13, 10)
    _, i, j = obsname.split("-")
    return int(i[1:]), int(j[1:])


rmax = max(r.abs().max() for _, _, r in residuals.values())
rnorm = mpl.colors.Normalize(vmin=-rmax, vmax=rmax)
values = np.concatenate([np.r_[t, s] for t, s, _ in residuals.values()])
lim = (np.floor(values.min()), np.ceil(values.max()))
markers = {"A": "o", "B": "s"}


def marker_size(r):
    # marker area (points²) grows with the absolute residual
    return 8.0 + 120.0 * np.abs(r) / rmax


with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A", "B", "C"], ["cb", "cb", "key"]],
        figsize=(width, 0.45 * width),
        layout="constrained",
        # C is square and the maps are 1.6 times taller than wide, so this
        # width makes C as tall as the maps
        width_ratios=[1.0, 1.0, 1.6],
        height_ratios=[1.0, 0.1],
    )
    for letter, (kind, label) in kinds.items():
        true, sim_head, r = residuals[letter]
        ax = axd[letter]
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
        mm.plot_array(head, masked_values=[1.0e30], vmin=vmin, vmax=vmax, alpha=0.4)
        mm.contour_array(
            lake.astype(float), levels=[0.5], colors="blue", linewidths=1.0
        )
        mm.plot_bc(package=gwf.sfr, color="cyan")
        cells = [obs_cell(n) for n in r.index]
        x = [gwf.modelgrid.xcellcenters[i, j] for i, j in cells]
        y = [gwf.modelgrid.ycellcenters[i, j] for i, j in cells]
        sc = ax.scatter(
            x,
            y,
            c=r,
            cmap="RdBu",
            norm=rnorm,
            marker=markers[letter],
            s=marker_size(r),
            edgecolors="black",
            linewidths=0.5,
            zorder=10,
        )
        ax.set_aspect("equal")
        ax.set_xlabel("x (m)")
        if letter == "A":
            ax.set_ylabel("y (m)")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=f"{label}")
    # the colorbar fills the middle of its row, under the two maps, and the key to
    # the marker sizes sits under C
    axd["cb"].axis("off")
    cax = axd["cb"].inset_axes([0.2, 0.55, 0.6, 0.3])
    fig.colorbar(
        sc, cax=cax, orientation="horizontal", label="Simulated minus true head (m)"
    )
    sizes = [0.05, 0.1, 0.2]
    axd["key"].axis("off")
    flopy.plot.styles.graph_legend(
        ax=axd["key"],
        handles=[
            axd["key"].scatter(
                [], [], s=marker_size(v), facecolor="white", edgecolors="black", lw=0.5
            )
            for v in sizes
        ],
        labels=[f"{v:g} m" for v in sizes],
        title="Absolute residual",
        loc="center",
        ncol=len(sizes),
        fontsize=7,
    )

    ax = axd["C"]
    ax.plot(lim, lim, color="0.5", lw=0.75, label="One-to-one line")
    for letter, (kind, label) in kinds.items():
        true, sim_head, _ = residuals[letter]
        ax.plot(
            true,
            sim_head,
            markers[letter],
            ms=4,
            mfc="none",
            color="black",
            label=label,
        )
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect("equal")
    flopy.plot.styles.heading(ax=ax, letter="C", heading="Head comparison")
    flopy.plot.styles.xlabel(ax=ax, label="True head (m)")
    flopy.plot.styles.ylabel(ax=ax, label="Simulated head (m)")
    flopy.plot.styles.graph_legend(ax=ax, loc="lower right", fontsize=7)

**What to look for.** The simulated heads are too low at every well: by 0.18 m on average at the water table and 0.19 m in the lower aquifer. The root-mean-square residuals are 0.19 and 0.20 m, and no well is off by more than 0.31 m. In the maps (A and B) the largest residuals, about 0.3 m, are at the two wells west of the stream in the middle of the valley, and the smallest, 0.04 m, is at the well nearest the coast in the southeast. On the one-to-one plot (C) every well lies within 0.31 m of the line.

The base model in the companion notebook misses by about twice as much, with root-mean-square residuals of 0.37 m at the water table and 0.33 m in the lower aquifer, so the advanced packages roughly halve the misfit. The residuals do not reach zero because the truth comes from the Voronoi-grid model, which has the same land surface, hydraulic conductivity, and unsaturated-zone properties but a different grid, with much finer cells along the stream (25 m) and over the lake (40 m). Halving the size of every cell in this model (80 rows and 50 columns) lowers the root-mean-square residuals only to 0.14 m at the water table and in the lower aquifer.

## Water budget

The listing file, `sv.lst`, reports the volumetric budget of the whole model at the end of every time step: the rate at which each package adds water to the aquifer (`IN`) and removes it (`OUT`), in the model units of cubic meters per day (m³/d). `gwf.output.list()` reads it, and `get_dataframes()` returns two pandas tables, the rates and the cumulative volumes, with one column per budget term and one row per time step. `start_datetime` labels the rows with dates; the model starts on January 1, 1962, so the steady-state period ends on January 1, 2006, and each transient period ends on January 1 of the following year.

A daily rate in cubic meters is hard to picture for a whole valley, so convert the rates to **million cubic meters per year** (million m³/yr), the metric counterpart of acre-feet per year: 1 million m³ is about 811 acre-feet.

The budget terms are named by package. `UZF-GWRCH` is the recharge UZF delivers to the water table, `LAK` and `SFR` are the leakage between the aquifer and the lake and the stream, and `MAW` is the pumping. The colors match the companion notebook where the two models share a term.

In [ ]:
budget, _ = gwf.output.list().get_dataframes(start_datetime="1962-01-01")

# m³/d to million m³/yr
M3D_TO_MCM_PER_YEAR = 365.25 / 1.0e6
budget = budget * M3D_TO_MCM_PER_YEAR

# budget terms, in a fixed order so each keeps its color in every figure
terms = {
    "UZF-GWRCH": "Recharge",
    "LAK": "Lake",
    "SFR": "Stream",
    "GHB": "Coast",
    "STO-SS": "Storage",
    "MAW": "Pumping wells",
}
colors = {
    "UZF-GWRCH": "#2a78d6",
    "LAK": "#eda100",
    "SFR": "#eb6834",
    "GHB": "#1baf7a",
    "STO-SS": "#e87ba4",
    "MAW": "#008300",
}

steady = budget.iloc[0]
for side in ("IN", "OUT"):
    rates = ", ".join(
        f"{label} {steady[f'{key}_{side}']:.2f}"
        for key, label in terms.items()
        if steady[f"{key}_{side}"] > 0
    )
    print(f"steady state {side.lower()}: {rates} million m³/yr")

### Steady state

At steady state the water entering the model equals the water leaving it. Draw the inflow (A) and the outflow (B) with `ax.pie()`, one slice per term. The lake and coast slices of the inflow are too small to label, so a legend under each pie gives each term's rate and share.

In [ ]:
sides = {"A": ("IN", "Inflow"), "B": ("OUT", "Outflow")}


def share(fraction):
    # one decimal for a share too small to round to a whole percent
    return f"{fraction:.0%}" if fraction >= 0.01 else f"{fraction:.1%}"


with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [list(sides)], figsize=(width, 0.55 * width), layout="constrained"
    )
    for letter, (side, heading) in sides.items():
        flows = {
            key: steady[f"{key}_{side}"] for key in terms if steady[f"{key}_{side}"] > 0
        }
        total = sum(flows.values())
        ax = axd[letter]
        wedges, _ = ax.pie(
            flows.values(),
            colors=[colors[key] for key in flows],
            startangle=90,
            counterclock=False,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5},
        )
        ax.set_aspect("equal")
        flopy.plot.styles.heading(
            ax=ax, letter=letter, heading=f"{heading}, {total:.1f} million m³/yr"
        )
        # small slices sit side by side, so name the slices in a legend below
        flopy.plot.styles.graph_legend(
            ax=ax,
            handles=wedges,
            labels=[
                f"{terms[key]}, {q:.2f} million m³/yr ({share(q / total)})"
                for key, q in flows.items()
            ],
            loc="upper center",
            bbox_to_anchor=(0.5, 0.0),
        )

**What to look for.** UZF recharge is 94 percent of the inflow (A), the lake leaks 6 percent into the aquifer, and seawater adds the last 0.1 percent along the coast. The stream drains 69 percent of the water (B), the coast 29 percent, and the aquifer discharges 2 percent into the lake. The budget has no evapotranspiration term: all of the evapotranspiration UZF simulates is taken from the unsaturated zone, before the water reaches the water table, so it is part of the UZF budget rather than this one. The drains remove no water, because the water table stays more than 0.15 m below the land surface away from the lake and the stream.

### Transient periods

Draw the budget of each year from stress period 2, the first year of pumping, to the end of the simulation as stacked bars with `ax.bar()`: the inflows stack upward from zero and the outflows downward, so in a year that balances the two stacks are the same height. `bottom` sets where each term's bar starts, on top of the terms already drawn. The lake below gets the same kind of figure, so `budget_bars()` draws the bars for any budget table whose columns are named `<term>_IN` and `<term>_OUT`. `flopy.plot.styles.USGSPlot()` sets the fonts and line weights, `heading()`, `xlabel()`, `ylabel()`, and `add_text()` label the figure in the same style, and `graph_legend()` draws the legend. Terms that are zero in every year are left out.

In [ ]:
def budget_bars(ax, table, terms, colors):
    """Stack the inflows of each year above zero and the outflows below it."""
    # each period ends on January 1, so it covers the year before its date
    years = np.array([date.year - 1 for date in table.index])
    handles = {}
    for side, sign in (("IN", 1.0), ("OUT", -1.0)):
        bottom = np.zeros(len(years))
        for key in terms:
            q = sign * table[f"{key}_{side}"].to_numpy()
            if (q == 0).all():
                continue
            bars = ax.bar(
                years,
                q,
                bottom=bottom,
                width=0.8,
                color=colors[key],
                edgecolor="white",
                linewidth=0.5,
            )
            handles.setdefault(key, bars)
            bottom += q
    # room above and below the tallest stack for the Inflow and Outflow labels
    ylim = 1.25 * table[["TOTAL_IN", "TOTAL_OUT"]].to_numpy().max()
    ax.set_ylim(-ylim, ylim)
    ax.axhline(0.0, color="black", lw=0.75)
    ax.set_xticks(years[::2])
    # outflows are drawn below zero, but label the axis with positive rates
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda y, pos: f"{abs(y):g}"))
    flopy.plot.styles.add_text(ax=ax, text="Inflow", x=0.01, y=0.97, va="top")
    flopy.plot.styles.add_text(ax=ax, text="Outflow", x=0.01, y=0.03, va="bottom")
    flopy.plot.styles.ylabel(ax=ax, label="Rate (million m³/yr)")
    # one legend entry for each term, in the order of terms
    keys = [key for key in terms if key in handles]
    flopy.plot.styles.graph_legend(
        ax=ax,
        handles=[handles[key] for key in keys],
        labels=[terms[key] for key in keys],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=4,
    )


transient = budget.iloc[1:]

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.7 * width), layout="constrained")
    budget_bars(ax, transient, terms, colors)
    flopy.plot.styles.heading(ax=ax, heading="Water budget by year")
    flopy.plot.styles.xlabel(ax=ax, label="Year")

In [ ]:
# the rates by the year each stress period covers, leaving out terms that are
# zero in every year
table = budget[[f"{key}_{side}" for side in ("IN", "OUT") for key in terms]]
table = table.loc[:, (table != 0).any()]
table.index = ["Steady state"] + [date.year - 1 for date in table.index[1:]]
table.round(2)

**What to look for.** The two wells start in 2006 and pump 3.46 million m³/yr. Recharge changes much more from year to year, from 9.71 million m³/yr in 2007 to 29.49 million m³/yr in 2018, a year of both the highest infiltration and the highest evapotranspiration demand in the UZF input. Most of each year's recharge leaves the same year: the stream drains 7.23 to 15.20 million m³/yr and the coast 2.74 to 7.37 million m³/yr, rising and falling with the recharge. Storage takes up some of the change: the aquifer releases as much as 1.79 million m³/yr from storage in a dry year (2007) and takes in as much as 1.94 million m³/yr in a wet one (2018). Pumping is still paid for mostly by less discharge to the stream and the coast rather than by a falling water table.

## The lake

LAK keeps its own water budget for the lake: rainfall on the lake, runoff and drain water the Water Mover brings to it (`FROM-MVR`), evaporation, leakage to and from the aquifer (`GWF`), and the change in the water stored in the lake. Read it from the CSV file the run saved with `gwf.lak.output.budgetcsv()`, whose `data` is a NumPy record array with one row per time step, and the stage from `gwf.lak.output.stage()`. In this model no runoff or drain water reaches the lake, because the water table stays well below the land surface around it, so `FROM-MVR` is zero and is left out of the figure.

The stream has its own budget too, but nearly all of it is groundwater discharge passing straight out to the sea, which the question below plots through time, so it is not repeated here.

In [ ]:
lake_budget = pd.DataFrame(gwf.lak.output.budgetcsv().data).set_index("totim")
# the same dates and units as the model budget
lake_budget.index = budget.index
lake_budget = lake_budget * M3D_TO_MCM_PER_YEAR

lake_terms = {
    "RAINFALL": "Rainfall",
    "FROM-MVR": "Runoff",
    "GWF": "Aquifer",
    "EVAPORATION": "Evaporation",
    "STORAGE": "Storage",
}
lake_colors = {
    "RAINFALL": "#2a78d6",
    "FROM-MVR": "#6250d6",
    "GWF": "#1baf7a",
    "EVAPORATION": "#eda100",
    "STORAGE": "#e87ba4",
}

stage_file = gwf.lak.output.stage()
lake_stage = stage_file.get_alldata().ravel()
print(
    f"lake stage {lake_stage[0]:.2f} m at steady state, "
    f"{lake_stage[1:].min():.2f} to {lake_stage[1:].max():.2f} m after"
)

lake_table = lake_budget[
    [f"{key}_{side}" for side in ("IN", "OUT") for key in lake_terms]
]
lake_table = lake_table.loc[:, (lake_table != 0).any()]
lake_table.index = ["Steady state"] + [date.year - 1 for date in lake_table.index[1:]]
lake_table.round(2)

Plot the stage at the end of each year (A) above the lake budget of each year (B), drawn with `budget_bars()`. On the year axis a bar is centered on the year it covers, so the stage at the end of that year sits halfway between it and the next bar.

In [ ]:
# the stage is reported at the end of each period, on January 1, which on the
# year axis is halfway between the bar of the year before and the next bar
stage_years = np.array([date.year - 0.5 for date in lake_budget.index])

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A"], ["B"]],
        figsize=(width, 0.95 * width),
        layout="constrained",
        sharex=True,
        height_ratios=[1.0, 2.2],
    )
    ax = axd["A"]
    ax.plot(stage_years, lake_stage, color="black", lw=1.5, marker="o", ms=3)
    flopy.plot.styles.heading(ax=ax, letter="A", heading="Lake stage")
    flopy.plot.styles.ylabel(ax=ax, label="Stage (m)")

    ax = axd["B"]
    budget_bars(ax, lake_budget.iloc[1:], lake_terms, lake_colors)
    flopy.plot.styles.heading(ax=ax, letter="B", heading="Lake budget by year")
    flopy.plot.styles.xlabel(ax=ax, label="Year")

**What to look for.** The stage falls through most of the simulation, from 5.97 m at steady state to 4.15 m at the end of 2025 (A), because in most years the lake loses more water to evaporation and to the aquifer than rainfall brings in, and the difference comes from the water stored in the lake (B). The lake leaks 0.67 to 2.49 million m³/yr to the aquifer in every year but 2018. The leakage is largest in 2007, a dry year with low recharge and a falling water table, and the stage falls 0.51 m that year, the most of any year. In 2018 rainfall is the highest of any year, 4.39 million m³/yr, and evaporation takes 5.19 million m³/yr, but the high recharge raises the water table around the lake above the stage, so the aquifer gives the lake 1.67 million m³/yr and the stage rises 0.29 m, to 4.86 m. From 2021 to 2025 rainfall stays below that of 2020, down to 1.64 million m³/yr in 2025, the lowest of any year, and the stage falls every year, to 4.15 m.

## Question: how does the well near the stream change the stream discharge?

The `vc` well pumps from the lower aquifer 457 m west of the stream. Some of the water it pumps would otherwise have discharged to the stream, so pumping lowers the flow the stream carries out of the model to the sea. That reduction is called **streamflow depletion**.

**What effect does pumping from the `vc` well have on the surface-water discharge from the model?** To answer it:

1. Run the notebook as it is, which records the discharge for the current `vc_rate`.
2. In the cell that sets the pumping rates, change `vc_rate`, for example to `0.0` (no pumping) or to twice the current rate.
3. Run that cell and every cell below it again. `build_advanced_without_maw()` does not need to run again, but the cells from **Build the Multi-Aquifer Well package** down do.
4. Repeat for as many rates as you like, then compare the runs in the figure at the end of this section.

`sfr_discharge()` reads the flow out of the last reach of the stream at the end of each stress period, in m³/d. `record_sfr_discharge()` keeps the runs in a dictionary whose key is the `vc` pumping rate and whose value is the discharge, replacing any earlier run at the same rate. It saves the dictionary to `models/sfr-discharge.pkl` with Python's `pickle` module, which writes a Python object to a binary file and reads it back unchanged, so the runs survive restarting the notebook. `pd.DataFrame(runs)` turns the dictionary into a table with one column per pumping rate.

In [ ]:
discharge = sfr_discharge(new_ws)
runs = record_sfr_discharge("models/sfr-discharge.pkl", vc_rate, discharge)
table = pd.DataFrame(runs)
table

Plot the stream discharge through time for each saved run (A), and the average discharge over the pumping period for each `vc` pumping rate (B). The time axis is the end of each stress period in years since the start of pumping.

In [ ]:
years = (table.index - table.index[0]) / 365.25
pumping = table.index > table.index[0]
mean_discharge = table[pumping].mean()

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]],
        figsize=(width, 0.4 * width),
        layout="constrained",
        width_ratios=[2.0, 1.0],
    )
    ax = axd["A"]
    for rate in table.columns:
        ax.plot(years, table[rate], marker="o", ms=3, label=f"{rate:g} m³/d")
    ax.set_xlim(years.min(), years.max())
    # room above the lines for the explanation
    ax.set_ylim(0.0, 1.4 * table.to_numpy().max())
    flopy.plot.styles.heading(ax=ax, letter="A", heading="Stream discharge to the sea")
    flopy.plot.styles.xlabel(ax=ax, label="Years since pumping started")
    flopy.plot.styles.ylabel(ax=ax, label="Discharge (m³/d)")
    flopy.plot.styles.graph_legend(
        ax=ax,
        loc="upper center",
        ncol=3,
        title="vc pumping rate",
        columnspacing=1.0,
        handlelength=1.5,
    )

    ax = axd["B"]
    ax.plot(-mean_discharge.index, mean_discharge, marker="o", color="black")
    ax.set_ylim(0.0, 1.1 * mean_discharge.max())
    flopy.plot.styles.heading(ax=ax, letter="B", heading="Average while pumping")
    flopy.plot.styles.xlabel(ax=ax, label="vc pumping (m³/d)")
    flopy.plot.styles.ylabel(ax=ax, label="Discharge (m³/d)")

**Hints for your answer.** Compare the average discharge in panel B for two pumping rates. The difference divided by the difference in pumping is the fraction of the pumped water that comes from the stream. Does that fraction change with the pumping rate? Does the stream take a year, or several, to feel a change in pumping (panel A)? Where else could the rest of the pumped water come from?

## Recap

You built the coastal synthetic-valley model with the UZF, SFR, LAK, and MVR packages, added the two pumping wells with the Multi-Aquifer Well package, ran the model, drew the steady-state water table in plan view and along two cross sections through the lake, compared the simulated heads at 17 observation wells to the true heads on a map and a one-to-one plot, and plotted the water budgets of the model and the lake. You then measured how the stream discharge to the sea changes with the pumping rate of the well near the stream.